# B2-024-gpu-scientific-ml-capstone — Practice p24 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** open-ended-experiment-design, open-ended-model-evaluation

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

TABLE = {
    "A": [0.0118, 0.0124, 0.0121, 0.0116, 0.0126],
    "B": [0.0109, 0.0113, 0.0112, 0.0105, 0.0116],
    "C": [0.0114, 0.0127, 0.0117, 0.0119, 0.0121],
    "D": [0.0161, 0.0148, 0.0172, 0.0155, 0.0166],
}

## Solution

### (a) Per-seed statistics and which differences are real

The cell below computes, for each configuration, the mean over seeds, and for B, C, D the five per-seed differences from A (same seed), their mean and their sample standard deviation (divisor $r-1=4$).

In [ ]:
import math

mean = lambda v: sum(v) / len(v)
sample_sd = lambda v: math.sqrt(sum((x - mean(v)) ** 2 for x in v) / (len(v) - 1))
MEANS = {c: mean(v) for c, v in TABLE.items()}
SDS = {c: sample_sd(v) for c, v in TABLE.items()}
DIFFS = {c: [b - a for a, b in zip(TABLE["A"], TABLE[c])] for c in "BCD"}
DIFF_STATS = {c: (mean(d), sample_sd(d)) for c, d in DIFFS.items()}
for c in "ABCD":
    print(f"{c}: mean {MEANS[c]:.5f}, seed sd {SDS[c]:.5f}")
for c in "BCD":
    m, s = DIFF_STATS[c]
    print(f"{c}-A per seed {[round(d, 4) for d in DIFFS[c]]}: mean {m:+.5f}, sample sd {s:.5f}, mean/sd {m / s:+.1f}, relative {m / MEANS['A']:+.1%}")

| vs A | mean | per-seed differences | mean diff | sample sd of diffs | paired seed-0 CI | verdict |
|---|---:|---|---:|---:|---|---|
| B | 0.01110 | −0.0009, −0.0011, −0.0009, −0.0011, −0.0010 | −0.00100 | 0.00010 | [−0.0014, −0.0004] | **real improvement** |
| C | 0.01196 | −0.0004, +0.0003, −0.0004, +0.0003, −0.0005 | −0.00014 | 0.00040 | [−0.0011, 0.0003] | **not demonstrated** |
| D | 0.01604 | +0.0043, +0.0024, +0.0051, +0.0039, +0.0040 | +0.00394 | 0.00098 | [0.0029, 0.0058] | **real degradation** |

* **B is real.** Evidence 1 (seed noise): all five shared-seed differences are negative and the mean difference is about ten sample standard deviations of the differences from zero. Evidence 2 (example noise): the paired seed-0 bootstrap interval over validation examples, $[-0.0014,-0.0004]$, excludes 0.
* **C is not demonstrated.** The per-seed differences change sign (three negative, two positive) and their mean, $-0.00014$, is a third of their standard deviation; the paired seed-0 interval $[-0.0011, 0.0003]$ contains 0. The wider network is indistinguishable from A at this budget.
* **D is real (worse).** All five differences are positive, with mean about four standard deviations above zero, and the paired interval $[0.0029,0.0058]$ excludes 0: removing the log transform hurts, by about 33%.

### (b) Overlapping separate intervals, paired interval excluding 0

The separate seed-0 intervals of A ($[0.0104,0.0134]$) and B ($[0.0096,0.0123]$) each contain the variation from *which* validation examples are resampled, and that variation is mostly shared: a resample full of hard (e.g. edge) sources raises both models' errors.
The paired interval resamples the same examples for both models, so the shared difficulty cancels in the per-example differences and only the much smaller variation in how the two models *differ* remains; B's error is lower than A's on most examples by a consistent small amount, so the paired interval is narrow and lies below 0.
Comparing two marginal intervals for overlap ignores this positive correlation and is far too conservative (practice p15: $\operatorname{Var}(\bar D)=2\sigma^2(1-\rho)/r$).

### (c) Adoption under the pre-registered 10% rule

B's mean relative reduction is $0.00100/0.01210=8.3\%$, below the pre-registered 10%. B's improvement is **real but not large enough**, so under the rule fixed before the runs **A stays the adopted configuration** (C is not demonstrated and D is worse). Lowering the bar now that the result is in would be exactly the after-the-fact threshold that pre-registration exists to prevent; B is recorded as the leading candidate for the next experiment.

### (d) Next experiments within 40 minutes

The most valuable open question is whether augmentation can clear the 10% bar, i.e. whether B's real 8% effect grows with a better-chosen noise level, since that is the only change that could alter the adoption decision.
Plan (32 runs ≈ 32 minutes, 8 minutes of slack for overruns): configurations A, B$_{0.03}$ (the current B), B$_{0.06}$, B$_{0.10}$ (noise std on the log magnitudes), each with 8 **new** shared seeds 5–12, all with the same 300 steps and the same validation rows; report each B variant's mean, the per-seed differences from A, and a paired bootstrap interval.
Decision change: if some noise level gives a mean reduction of at least 10% relative to A, with all or nearly all shared-seed differences negative and a paired interval excluding 0, I adopt that configuration; if all B variants stay below 10% (or the larger noise starts to hurt), I keep A and stop pursuing augmentation. (Because the noise level is now chosen on validation, its best validation gain is slightly optimistic; the test set still sees only one final call.)

### (e) Hidden-test discipline

Scoring B and C on the locked test set to "break the tie" turns the test set into a selection set: the chosen model is the better of two noisy test evaluations, so its reported test score is optimistically biased and no longer an unbiased estimate, and the decision itself would be made on 200 examples whose noise the paired analysis shows is comparable to the effects in question.
It also breaks the one-call rule (`test_call_count` would be 2).
Instead, the tie (B versus C) is broken on validation by the analysis above (B real, C not), the 10% rule decides adoption (A), and if more evidence is needed it comes from more seeds or a larger validation analysis; the test set is scored exactly once, for the final adopted configuration, after all choices are made.

### Answer check

In [ ]:
assert abs(MEANS["A"] - 0.0121) < 1e-12 and abs(MEANS["B"] - 0.0111) < 1e-12
assert abs(MEANS["C"] - 0.01196) < 1e-12 and abs(MEANS["D"] - 0.01604) < 1e-12
assert all(d < 0 for d in DIFFS["B"]) and all(d > 0 for d in DIFFS["D"])
assert any(d > 0 for d in DIFFS["C"]) and any(d < 0 for d in DIFFS["C"])
assert abs(DIFF_STATS["B"][0] + 0.0010) < 1e-12 and abs(DIFF_STATS["C"][0] + 0.00014) < 1e-12 and abs(DIFF_STATS["D"][0] - 0.00394) < 1e-12
assert abs(DIFF_STATS["B"][0]) > 5 * DIFF_STATS["B"][1]
assert abs(DIFF_STATS["C"][0]) < DIFF_STATS["C"][1]
relative_B = -DIFF_STATS["B"][0] / MEANS["A"]
assert 0.08 < relative_B < 0.10                      # real but below the 10% bar -> keep A
ADOPTED = "A" if relative_B < 0.10 else "B"
assert ADOPTED == "A"